In [1]:
!pip install -q transformers sentencepiece

In [3]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", device)

Device: cuda


In [4]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import torch.nn.functional as F

nli_name = "cross-encoder/nli-deberta-v3-base"

nli_tokenizer = AutoTokenizer.from_pretrained(nli_name)
nli_model = AutoModelForSequenceClassification.from_pretrained(nli_name)

nli_model = nli_model.to(device)
nli_model.eval()

print(nli_model.config.id2label)

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

{0: 'contradiction', 1: 'entailment', 2: 'neutral'}


In [5]:
def verify_claim(claim, evidence):

    inputs = nli_tokenizer(
        evidence,
        claim,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=512
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = nli_model(**inputs)

    probabilities = F.softmax(outputs.logits, dim=1)[0]

    labels = nli_model.config.id2label

    results = {
        labels[i].lower(): probabilities[i].item()
        for i in range(len(labels))
    }

    return results

In [6]:
claim = "Vitamin D supplementation reduces respiratory infections."

evidence = """
Vitamin D supplementation was associated with a reduced risk
of acute respiratory infections in the studied population.
"""

result = verify_claim(claim, evidence)

print(result)

{'contradiction': 0.0001033765947795473, 'entailment': 0.9950319528579712, 'neutral': 0.004864695481956005}


In [7]:
claim = "Vitamin D supplementation reduces respiratory infections."

evidence = """
The study found no reduction in respiratory infection risk
following vitamin D supplementation.
"""

print(verify_claim(claim, evidence))

{'contradiction': 0.9993267059326172, 'entailment': 0.0001652306382311508, 'neutral': 0.0005080790724605322}
